# willmarley: battery runtime

Tracking issues: [#85](https://github.com/MarMar888/hav/issues/85) (this repo) and [#1562](https://github.com/MarMar888/lot/issues/1562).

One equation:

`Run time (min) = [0.8 * Capacity (Ah) * Voltage (V) * Battery efficiency] / [Load power (W) * Motor efficiency] * 60 min/h`

Capacity times voltage is the pack's energy in watt-hours. Only 80% of it is used. Usable energy divided by power is time.

| Symbol | Meaning | Unit |
|---|---|---|
| `C` | battery capacity | Ah |
| `V` | pack voltage | V |
| `eta_b` | battery efficiency | 0 to 1 |
| `P` | load power | W |
| `eta_m` | motor efficiency | 0 to 1 |

**Heads up:** `eta_m` divides, exactly as the equation is written, so a higher motor efficiency gives a *shorter* run time here. The usual convention is the opposite. If that is not what you meant, change the one line in `runtime` below.

## The equation and the bounds

Each input gets a lower and an upper bound. Edit them. The midpoint of each range is the baseline.

In [ ]:
%pip install -q matplotlib numpy
import matplotlib.pyplot as plt
import numpy as np

USABLE = 0.8  # fraction of the battery used

# (lower, upper) for every input. Edit these.
BOUNDS = {
    "C":     (10.0, 50.0),    # Ah
    "V":     (11.1, 25.2),    # V, a 3S to a 6S pack
    "eta_b": (0.90, 0.98),    # battery efficiency
    "P":     (100.0, 500.0),  # W
    "eta_m": (0.70, 0.90),    # motor efficiency
}
LABELS = {
    "C": "battery capacity (Ah)",
    "V": "pack voltage (V)",
    "eta_b": "battery efficiency",
    "P": "load power (W)",
    "eta_m": "motor efficiency",
}

MID = {k: (lo + hi) / 2 for k, (lo, hi) in BOUNDS.items()}


def runtime(C, V, eta_b, P, eta_m):
    """Run time in minutes."""
    return 60 * USABLE * C * V * eta_b / (P * eta_m)


print("All inputs at the midpoint of their bounds:")
for k, v in MID.items():
    print(f"  {k:>5} = {v:g}")
print(f"\nRuntime {runtime(**MID):.1f} minutes")

## Sweep each input on its own

Hold every other input at its midpoint, move one across its bounds. A steeper line means that input matters more.

In [ ]:
fig, axes = plt.subplots(1, len(BOUNDS), figsize=(18, 3.6), sharey=True)
for ax, (name, (lo, hi)) in zip(axes, BOUNDS.items()):
    xs = np.linspace(lo, hi, 60)
    ax.plot(xs, [runtime(**{**MID, name: x}) for x in xs])
    ax.axvline(MID[name], color="grey", ls=":", lw=1)
    ax.set(xlabel=LABELS[name], title=name)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("runtime (min)")
fig.suptitle("Runtime as each input moves alone", y=1.0)
plt.tight_layout()
plt.show()

for name, (lo, hi) in BOUNDS.items():
    a, b = runtime(**{**MID, name: lo}), runtime(**{**MID, name: hi})
    print(f"{name:>5}: {a:6.1f} min at lower bound, {b:6.1f} min at upper bound")

## SSS motors: torque vs speed (first quadrant)

First quadrant means positive speed and positive torque, the motor driving the load. The numbers come from the SSS spec sheet (`SSS-Specs-2.pdf`, 12 pages, 27 motor models, 263 windings), typed into `sss_motors.csv` next to this notebook (27 motor models, 263 windings). The sheet lists no torque curve, so each line is built from three equations:

- `speed (rpm) = Kv * (V - I * R)`: the motor spins up until its back-voltage balances what is left after the resistive drop.
- `Kt = 60 / (2 * pi * Kv)`  (N*m per amp): the torque constant follows from `Kv`.
- `torque (N*m) = Kt * (I - I0)`: torque comes from the current beyond what the motor draws spinning freely.

**Why the line has a flat start.** At zero speed there is no back-voltage, so the full voltage across the tiny resistance `R` would push `V / R` amps (thousands, and tens of N*m). The sheet's max current stops that, so the torque is capped at `Kt * (I_max - I0)`. Each line therefore starts at rpm = 0 at that maximum torque and stays flat while the controller holds the current at `I_max`. It bends down once the voltage runs out (`V_max = I_max * R + back-voltage`, at the "corner" speed), then falls straight to zero torque at the no-load speed.

Every winding is drawn at its own rated voltage. Windings are named as on the sheet: `Y` or `D` (star or delta) after the turn count.

Assumptions: `R` is the sheet's resistance column taken as ohms. `I0` is the sheet's no-load current, measured at 11.1 V, held constant.

**Suspect resistance values.** Many resistance entries on the sheet look like typos with a digit dropped (for example 3656-V2 5Y lists 0.083 ohm, but its neighbour lists 0.013). A line is drawn *dotted* when its max current times its resistance is under 1% or over 30% of its max voltage, because that puts the corner speed in an implausible place. The sheet's value is used as printed. Only the corner speed depends on `R`; the flat torque and the no-load speed do not.

In [ ]:
import csv
from pathlib import Path

CSV = next(p for p in (Path("sss_motors.csv"), Path("optimization/sss_motors.csv")) if p.exists())

# motor -> {"windings": [...], "p_cont", "p_max", "max_rpm"}, read from the spec sheet
MOTORS = {}
with open(CSV) as f:
    for r in csv.DictReader(f):
        m = MOTORS.setdefault(r["motor"], {"windings": [], "p_cont": float(r["p_cont_w"]),
                                          "p_max": float(r["p_max_w"]), "max_rpm": float(r["max_rpm"])})
        m["windings"].append(dict(name=r["winding"], Kv=float(r["kv_rpm_per_v"]), I_max=float(r["i_max_a"]),
                                  V_max=float(r["v_max_v"]), R=float(r["r_ohm"]), I0=float(r["i0_a_at_11v"])))


def curve(w):
    """(rpm, torque) corners of one winding's line, and whether its R looks like a typo."""
    Kt = 60 / (2 * np.pi * w["Kv"])
    T_max = Kt * (w["I_max"] - w["I0"])                  # torque at the current limit
    rpm_corner = w["Kv"] * (w["V_max"] - w["I_max"] * w["R"])  # speed where the voltage runs out
    rpm_free = w["Kv"] * (w["V_max"] - w["I0"] * w["R"])       # no-load speed
    drop = w["I_max"] * w["R"] / w["V_max"]
    return [0, rpm_corner, rpm_free], [T_max, T_max, 0], not 0.01 <= drop <= 0.30


### All 5Y windings on one graph, with the motors earlier PEP teams used

Every motor that has a 5Y winding, one line per motor, torque against speed. 5Y has a different Kv, voltage and current on each motor, which is why the lines differ. Set `WINDING` to look at another winding.

**Highlighted (thick, black outline, star in the legend):** SSS motors named in the PEP 2025 and 2026 white papers (`PEP2025TeamExtractions/`, `PEP2026TeamExtractions/`). Only three motor models appear:

| Motor | Teams | Winding the paper gives |
|---|---|---|
| 5684-V2 | Alabama 2025 and 2026 | 1050 Kv, 165 A, 45 V, which is the **5Y** winding |
| 56123-V2 | VCU 2025 and 2026; Illinois 2025 and 2026; Texas A&M 2026 | VCU: 500 Kv, which is the **6Y**. Illinois 2025 says 550 Kv and Texas A&M says 700 Kv, neither of which is a 56123-V2 winding on the sheet. Illinois 2026 gives no Kv. |
| 56154-V2 | Auburn 2025; East Carolina 2026 | none stated (Auburn says 25,000 rpm at 55.5 V) |

So only 5684-V2 is confirmed to have used its 5Y winding. For the other two the 5Y line shows the same motor, not necessarily the same winding. Other teams used non-SSS motors (for example NC State's TP5680 and Georgia's KingVal 4092) and are not drawn.

In [ ]:
WINDING = "5Y"

# SSS motors named in the PEP 2025 / 2026 white papers -> teams that used them
USED_BY = {
    "SSS 5684-V2": "Alabama '25 '26",
    "SSS 56123-V2": "VCU '25 '26, Illinois '25 '26, Texas A&M '26",
    "SSS 56154-V2": "Auburn '25, ECU '26",
}

picked = [(motor, w) for motor, d in MOTORS.items() for w in d["windings"] if w["name"] == WINDING]
others = iter(plt.cm.turbo(np.linspace(0.02, 0.98, len(picked))))
USED_COLOR = {"SSS 5684-V2": "tab:blue", "SSS 56123-V2": "tab:red", "SSS 56154-V2": "tab:green"}

fig, ax = plt.subplots(figsize=(15, 7.5))
for motor, w in picked:
    rpm, torque, suspect = curve(w)
    used = motor in USED_BY
    color = USED_COLOR[motor] if used else next(others)
    label = f"{motor.replace('SSS ', '')}  ({w['Kv']:g} Kv, {w['V_max']:g} V, {w['I_max']:g} A)"
    if used:
        ax.plot(rpm, torque, color="black", lw=6, solid_capstyle="round", zorder=2)
        label = f"★ {label}  – {USED_BY[motor]}"
    ax.plot(rpm, torque, color=color, lw=3.5 if used else 1.2, alpha=1 if used else 0.55,
            ls=":" if suspect else "-", label=label, zorder=3 if used else 1)
ax.set(xlabel="speed (rpm)", ylabel="torque (N*m)", xlim=(0, None), ylim=(0, None),
       title=f"{WINDING} windings of every SSS motor: torque vs speed\n"
             f"★ thick = motor used by a PEP 2025 / 2026 team   dotted = suspect resistance on the sheet")
ax.grid(alpha=0.3)
ax.legend(title="motor", fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()